In [1]:
print("Sydney PM2.5 Capstone")
print("Python environment is working!")

Sydney PM2.5 Capstone
Python environment is working!


In [2]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 3.0.5
NumPy version: 2.5.2


#### 1. Data Inspection

This notebook is used to inspect the raw NSW air quality datasets before data cleaning, merging and modelling.

##### 1.1 Project and Data Directory Setup
The project and raw data directories are identified to ensure that the analysis uses the correct source files.

In [3]:
from pathlib import Path

project_folder = Path.cwd().parent
raw_data_folder = project_folder / "data" / "raw"

print("Project folder:")
print(project_folder)

print("\nRaw data folder:")
print(raw_data_folder)

Project folder:
c:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone

Raw data folder:
c:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone\data\raw


In [4]:
# Check if the raw data folder exists
print("Does the raw data folder exist?")
print(raw_data_folder.exists())

Does the raw data folder exist?
True


##### 1.2 Station and File Inventory
The raw data directory is examined to identify the available monitoring stations and the files downloaded for each station.

In [5]:
#Finding the CSV files in the raw data folder of the stations
station_folders = [folder for folder in raw_data_folder.iterdir() if folder.is_dir()]

print("Number of station folders found:", len(station_folders))
print("\nStation folders:")

for folder in station_folders:
    print("-", folder.name)


Number of station folders found: 9

Station folders:
- Camden
- Lindfield
- Liverpool
- Parramatta North
- Penrith
- Prospect
- Randwick
- Rozelle
- Ultimo-UTS


In [6]:
#Finding the CSV files in each station folder (the features of the stations)
for station_folder in station_folders:
    print(f"\n{station_folder.name}")
    print("-" * len(station_folder.name))
    
    files = list(station_folder.iterdir())
    
    for file in files:
        if file.is_file():
            print(" ", file.name)


Camden
------
  Air Temperature, Relative Humidity.xls
  NO2, CO, SO2.xls
  Ozone, NO.xls
  PM10, PM2.5, Ammonia - Copy.xlsx
  PM10, PM2.5, Ammonia.xls
  Rainfall.xls
  Wind Speed, Wind Direction.xls

Lindfield
---------
  Air Temperature, Relative Humidity, Rainfall.xls
  NO2, CO, SO2.xls
  Ozone, NO.xls
  PM10, PM2.5, Ammonia.xls
  Wind Speed, Wind Direction.xls

Liverpool
---------
  Air Temperature, Relative Humidity.xls
  NO2, CO.xls
  Ozone, NO.xls
  PM2.5, Ammonia.xls
  Rainfall.xls
  SO2, PM10.xls
  Wind Speed, Wind Direction.xls

Parramatta North
----------------
  Air Temperature, Relative Humidity.xls
  NO2, CO.xls
  Ozone, NO.xls
  PM2.5, Ammonia - Copy.xlsx
  PM2.5, Ammonia.xls
  Rainfall.xls
  SO2, PM10 - Copy.xlsx
  SO2, PM10.xls
  Wind speed, Wind Direction.xls

Penrith
-------
  Air Temperature, Relative humidity.xls
  NO2, CO.xls
  Ozone, NO.xls
  PM2.5, Ammonia.xls
  Rainfall.xls
  SO2, PM10.xls
  Wind Speed, Wind Direction.xls

Prospect
--------
  Air Temperature, 

In [7]:
#Create Summary of the CSV files in each station
station_summary = []

for station_folder in station_folders:
    files = [file for file in station_folder.iterdir() if file.is_file()]
    
    station_summary.append({
        "Station": station_folder.name,
        "Number of files": len(files)
    })

station_summary_df = pd.DataFrame(station_summary)

station_summary_df

,Station,Number of files
0,Camden,7
1,Lindfield,5
2,Liverpool,7
3,Parramatta North,9
4,Penrith,7
5,Prospect,6
6,Randwick,6
7,Rozelle,7
8,Ultimo-UTS,6


In [8]:
#Make complete Inventory table of all CSV files in each station folder
file_inventory = []

for station_folder in station_folders:
    for file in station_folder.iterdir():
        if file.is_file():
            file_inventory.append({
                "Station": station_folder.name,
                "File": file.name
            })

file_inventory_df = pd.DataFrame(file_inventory)

file_inventory_df

,Station,File
0,Camden,"Air Temperature, Relative Humidity.xls"
1,Camden,"NO2, CO, SO2.xls"
2,Camden,"Ozone, NO.xls"
3,Camden,"PM10, PM2.5, Ammonia - Copy.xlsx"
4,Camden,"PM10, PM2.5, Ammonia.xls"
5,Camden,Rainfall.xls
6,Camden,"Wind Speed, Wind Direction.xls"
7,Lindfield,"Air Temperature, Relative Humidity, Rainfall.xls"
8,Lindfield,"NO2, CO, SO2.xls"
9,Lindfield,"Ozone, NO.xls"


In [9]:
#Save the inventory table to "tables" under "results"
inventory_output = project_folder / "results" / "tables" / "raw_data_inventory.csv"

file_inventory_df.to_csv(inventory_output, index=False)

print("Inventory saved to:")
print(inventory_output)

Inventory saved to:
c:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone\results\tables\raw_data_inventory.csv


In [10]:
#Did this to find out the exact filename of the file 
#conatining the target variable (PM2.5) 
# in the Parramatta North station folder

file_inventory_df

,Station,File
0,Camden,"Air Temperature, Relative Humidity.xls"
1,Camden,"NO2, CO, SO2.xls"
2,Camden,"Ozone, NO.xls"
3,Camden,"PM10, PM2.5, Ammonia - Copy.xlsx"
4,Camden,"PM10, PM2.5, Ammonia.xls"
5,Camden,Rainfall.xls
6,Camden,"Wind Speed, Wind Direction.xls"
7,Lindfield,"Air Temperature, Relative Humidity, Rainfall.xls"
8,Lindfield,"NO2, CO, SO2.xls"
9,Lindfield,"Ozone, NO.xls"


##### 1.3 Initial Inspection of Individual Dataset

A representative dataset from Parramatta North is examined in detail to understand the structure of the downloaded NSW air quality files before developing automated data-quality checks.

In [11]:
#Locate the Parramatta North station folder 
# and list all files stored inside it

parramatta_folder = raw_data_folder / "PARRAMATTA NORTH"

print("Files in Parramatta North:")
for file in parramatta_folder.iterdir():
    if file.is_file():
        print(file.name)

Files in Parramatta North:
Air Temperature, Relative Humidity.xls
NO2, CO.xls
Ozone, NO.xls
PM2.5, Ammonia - Copy.xlsx
PM2.5, Ammonia.xls
Rainfall.xls
SO2, PM10 - Copy.xlsx
SO2, PM10.xls
Wind speed, Wind Direction.xls


##### 1.4 Data Structure and Format (Opening ONE Excel file with Python)

The PM2.5 dataset is examined to identify its file structure, column names, time fields and organisation of observations. This step is performed before data preprocessing so that the appropriate methods can be selected based on the actual dataset structure.

In [12]:
#Get exact file path for "PM2.5, Ammonia".xlsx file
# since PM2.5 is our target variable
pm25_file = parramatta_folder / "PM2.5, Ammonia - copy.xlsx"

print("File:")
print(pm25_file)
print("\nDoes this file exist?")
print(pm25_file.exists())


File:
c:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone\data\raw\PARRAMATTA NORTH\PM2.5, Ammonia - copy.xlsx

Does this file exist?
True


In [13]:
#Read the "PM2.5, Ammonia".xlsx file into a DataFrame
pm25_raw = pd.read_excel(pm25_file)

print("Rows:", len(pm25_raw))
print("Columns:", len(pm25_raw.columns))

print("\nColumn names:")
for column in pm25_raw.columns:
    print("-", column)

Rows: 57386
Columns: 3

Column names:
- Hourly Averages Time Range: 01/01/2020 00:00 to 19/07/2026 00:00
- Unnamed: 1
- Unnamed: 2


In [14]:
#Display the first few rows of the DataFrame
pm25_raw.head()

,Hourly Averages Time Range: 01/01/2020 00:00 to 19/07/2026 00:00,Unnamed: 1,Unnamed: 2
0,Initial Data,NaN,NaN
1,Date,Time,PARRAMATTA NORTH PM2.5 1h average [µg/m³]
2,01/01/2020,01:00,13.2
3,01/01/2020,02:00,9.3
4,01/01/2020,03:00,7.4


##### 1.5 Data Loading and Header Identification
The downloaded NSW air quality spreadsheets contain metadata and introductory rows before the actual observation table. Therefore, the position of the header row must be identified before the dataset can be loaded for analysis. The PM2.5 dataset indicates that the observation headers begin with the Date, Time and PM2.5 measurement fields. The data will be loaded using these identified headers while retaining the original raw file unchanged.

In [15]:
#Reading the "PM2.5, Ammonia".xlsx file into a DataFrame with header row specified
pm25_raw = pd.read_excel(
    pm25_file,
    header=2
)

print("Rows:", len(pm25_raw))
print("Columns:", len(pm25_raw.columns))

print("\nColumn names:")
for column in pm25_raw.columns:
    print("-", column)

#Display the first few rows of the new DataFrame with specified header row
pm25_raw.head()

Rows: 57384
Columns: 3

Column names:
- Date
- Time
- PARRAMATTA NORTH PM2.5 1h average [µg/m³]


,Date,Time,PARRAMATTA NORTH PM2.5 1h average [µg/m³]
0,01/01/2020,01:00,13.2
1,01/01/2020,02:00,9.3
2,01/01/2020,03:00,7.4
3,01/01/2020,04:00,12.3
4,01/01/2020,05:00,19.4


##### 1.6 Date, Time and Variable Validation

The date and time fields are examined to determine whether the observations are recorded at the expected hourly frequency. The data types of each column are also checked because the downloaded spreadsheet may initially store dates, times and measurements as text rather than numerical or datetime values. Correct identification and conversion of these fields is necessary before performing time-series analysis, missing-value assessment and model development.

In [16]:
#Checking data types of the variables in the DataFrame
pm25_raw.dtypes

#The Date and Time are stored as text.
#We got to convert them into datetime format for better analysis later on.

Date                                             str
Time                                             str
PARRAMATTA NORTH PM2.5 1h average [µg/m³]    float64
dtype: object

In [17]:
#Check first and last observations in the DataFrame
print("First observation:")
print(pm25_raw.iloc[0])

print("\nLast observation:")
print(pm25_raw.iloc[-1])

First observation:
Date                                         01/01/2020
Time                                              01:00
PARRAMATTA NORTH PM2.5 1h average [µg/m³]          13.2
Name: 0, dtype: object

Last observation:
Date                                         18/07/2026
Time                                              24:00
PARRAMATTA NORTH PM2.5 1h average [µg/m³]           8.5
Name: 57383, dtype: object


##### 1.7 Missing Value Assessment

Missing observations will be assessed before selecting an imputation strategy. The number and proportion of missing values will first be calculated for each variable. The temporal distribution of missing observations will also be examined because short isolated gaps may be suitable for interpolation, whereas long or systematic gaps may require a different treatment. The final missing-value strategy will therefore be based on the observed characteristics of the dataset rather than being predetermined.

In [18]:
#Count missing values in the DataFrame
missing_summary = pd.DataFrame({
    "Missing Count": pm25_raw.isna().sum(),
    "Total Observations": len(pm25_raw),
    "Missing Percentage": (pm25_raw.isna().sum() / len(pm25_raw)) * 100
})

missing_summary

,Missing Count,Total Observations,Missing Percentage
Date,0,57384,0.000000
Time,0,57384,0.000000
PARRAMATTA NORTH PM2.5 1h average [µg/m³],2393,57384,4.170152


##### 1.8 Missing Value Pattern

The proportion of missing PM2.5 observations was first calculated to assess the overall completeness of the dataset. However, the percentage of missing values alone does not indicate whether interpolation is appropriate. Therefore, the distribution and consecutive length of missing periods will be examined. Short, isolated gaps may be suitable for interpolation, whereas prolonged gaps may indicate periods of unavailable monitoring data and may require removal or retention rather than interpolation.

In [19]:
#Find out where missing values are located in the DataFrame
pm25_missing = pm25_raw[
    pm25_raw["PARRAMATTA NORTH PM2.5 1h average [µg/m³]"].isna()
].copy()

print("Number of missing PM2.5 observations:", len(pm25_missing))

print("\nFirst 20 missing observations:")
pm25_missing.head(20)

Number of missing PM2.5 observations: 2393

First 20 missing observations:


,Date,Time,PARRAMATTA NORTH PM2.5 1h average [µg/m³]
102,05/01/2020,07:00,NaN
547,23/01/2020,20:00,NaN
808,03/02/2020,17:00,NaN
809,03/02/2020,18:00,NaN
874,06/02/2020,11:00,NaN
875,06/02/2020,12:00,NaN
1404,28/02/2020,13:00,NaN
1479,02/03/2020,16:00,NaN
1480,02/03/2020,17:00,NaN
1856,18/03/2020,09:00,NaN


In [20]:
#Creating proper timestamp (datatime) column 
pm25_check = pm25_raw.copy()

# Convert the Date column using the Australian day/month/year format
pm25_check["Date"] = pd.to_datetime(
    pm25_check["Date"],
    format="%d/%m/%Y"
)

pm25_check["Time"] = pm25_check["Time"].astype(str)

# Handle 24:00 separately because pandas does not accept 24 as an hour
midnight_24 = pm25_check["Time"] == "24:00"

pm25_check.loc[midnight_24, "Time"] = "00:00"

pm25_check.loc[midnight_24, "Date"] = (
    pm25_check.loc[midnight_24, "Date"] + pd.Timedelta(days=1)
)

# Combine Date and Time into one datetime column
pm25_check["DateTime"] = pd.to_datetime(
    pm25_check["Date"].dt.strftime("%d/%m/%Y") + " " +
    pm25_check["Time"],
    format="%d/%m/%Y %H:%M"
)

print("First DateTime:")
print(pm25_check["DateTime"].iloc[0])

print("\nLast DateTime:")
print(pm25_check["DateTime"].iloc[-1])

First DateTime:
2020-01-01 01:00:00

Last DateTime:
2026-07-19 00:00:00


In [21]:
#Find consecutive missing values in the PM2.5 column
pm25_col = "PARRAMATTA NORTH PM2.5 1h average [µg/m³]"

is_missing = pm25_check[pm25_col].isna()

# Identify consecutive groups of missing and non-missing observations
groups = is_missing.ne(is_missing.shift()).cumsum()

missing_runs = (
    pm25_check[is_missing]
    .groupby(groups[is_missing])
    .agg(
        Start=("DateTime", "first"),
        End=("DateTime", "last"),
        Missing_Hours=("DateTime", "size")
    )
    .reset_index(drop=True)
)

missing_runs = missing_runs.sort_values(
    "Missing_Hours",
    ascending=False
)

missing_runs.head(20)

,Start,End,Missing_Hours
184,2023-12-07 05:00:00,2024-01-17 11:00:00,991
16,2020-06-12 10:00:00,2020-06-19 17:00:00,176
330,2025-12-26 19:00:00,2025-12-31 12:00:00,114
230,2024-08-12 11:00:00,2024-08-15 22:00:00,84
38,2020-11-14 13:00:00,2020-11-16 16:00:00,52
386,2026-06-01 16:00:00,2026-06-03 16:00:00,49
262,2024-12-11 10:00:00,2024-12-13 08:00:00,47
15,2020-05-19 21:00:00,2020-05-21 17:00:00,45
259,2024-11-26 21:00:00,2024-11-28 08:00:00,36
140,2023-05-24 09:00:00,2023-05-25 17:00:00,33


In [22]:
#Summarize the missing runs
print("Number of missing periods:", len(missing_runs))

print("\nTotal missing observations:", missing_runs["Missing_Hours"].sum())

print("\nMissing-run statistics:")
print(missing_runs["Missing_Hours"].describe())


Number of missing periods: 413

Total missing observations: 2393

Missing-run statistics:
count    413.000000
mean       5.794189
std       50.147683
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max      991.000000
Name: Missing_Hours, dtype: float64


##### 1.9 Missing Gap Distribution

The missing-value analysis identified 2,393 missing PM2.5 observations across 413 separate missing periods. Most missing periods were short, with a median duration of one hour and 75% lasting two hours or less. However, several substantially longer gaps were also identified, including a maximum gap of 991 consecutive hours. Therefore, missing observations will not be treated uniformly. The distribution of gap lengths will be examined to determine an appropriate strategy for short gaps, while prolonged gaps will be treated separately to avoid introducing unreliable interpolated values.

In [23]:
#Quantifying the different types of gaps
gap_summary = pd.DataFrame({
    "Gap length": ["1 hour", "2 hours", "3–6 hours", "7–24 hours", "25–168 hours", ">168 hours"],
    "Number of gaps": [
        (missing_runs["Missing_Hours"] == 1).sum(),
        (missing_runs["Missing_Hours"] == 2).sum(),
        missing_runs["Missing_Hours"].between(3, 6).sum(),
        missing_runs["Missing_Hours"].between(7, 24).sum(),
        missing_runs["Missing_Hours"].between(25, 168).sum(),
        (missing_runs["Missing_Hours"] > 168).sum()
    ]
})

gap_summary

,Gap length,Number of gaps
0,1 hour,286
1,2 hours,72
2,3–6 hours,29
3,7–24 hours,13
4,25–168 hours,11
5,>168 hours,2


In [24]:
#Calculate the total number of missing observations for each gap length category
observation_summary = pd.DataFrame({
    "Gap length": ["1 hour", "2 hours", "3–6 hours", "7–24 hours", "25–168 hours", ">168 hours"],
    "Missing observations": [
        missing_runs.loc[missing_runs["Missing_Hours"] == 1, "Missing_Hours"].sum(),
        missing_runs.loc[missing_runs["Missing_Hours"] == 2, "Missing_Hours"].sum(),
        missing_runs.loc[missing_runs["Missing_Hours"].between(3, 6), "Missing_Hours"].sum(),
        missing_runs.loc[missing_runs["Missing_Hours"].between(7, 24), "Missing_Hours"].sum(),
        missing_runs.loc[missing_runs["Missing_Hours"].between(25, 168), "Missing_Hours"].sum(),
        missing_runs.loc[missing_runs["Missing_Hours"] > 168, "Missing_Hours"].sum()
    ]
})

observation_summary["Percentage of missing observations"] = (
    observation_summary["Missing observations"]
    / observation_summary["Missing observations"].sum()
    * 100
)

observation_summary

,Gap length,Missing observations,Percentage of missing observations
0,1 hour,286,11.951525
1,2 hours,144,6.017551
2,3–6 hours,110,4.596740
3,7–24 hours,143,5.975763
4,25–168 hours,543,22.691183
5,>168 hours,1167,48.767238


##### 1.10 Cross-Variable Missing Data Assessment (PM10 Analysis for Parramatta North)

The PM10 dataset is examined alongside PM2.5 to determine whether prolonged PM2.5 gaps correspond to broader periods of missing observations at the monitoring station. Comparing variables recorded at the same timestamps helps distinguish pollutant-specific missing data from possible station-wide monitoring outages.

###### Repeating the same thing for PM10 of Parramatta North


In [25]:
#Finding the file containing PM10 data inParramatta North station folder
for file in parramatta_folder.iterdir():
    if "PM10" in file.name:
        print(file.name)

SO2, PM10 - Copy.xlsx
SO2, PM10.xls


In [26]:
#Check if the Copy of the PM10 file exists
pm10_file = parramatta_folder / "SO2, PM10 - Copy.xlsx"

print("File:")
print(pm10_file)

print("\nDoes the file exist?")
print(pm10_file.exists())

File:
c:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone\data\raw\PARRAMATTA NORTH\SO2, PM10 - Copy.xlsx

Does the file exist?
True


In [27]:
#Inspect the PM10 file structure
pm10_raw = pd.read_excel(
    pm10_file,
    header=2
)

print("Rows:", len(pm10_raw))
print("Columns:", len(pm10_raw.columns))

print("\nColumn names:")
for column in pm10_raw.columns:
    print("-", column)

Rows: 57384
Columns: 4

Column names:
- Date
- Time
- PARRAMATTA NORTH SO2 1h average [pphm]
- PARRAMATTA NORTH PM10 1h average [µg/m³]


In [28]:
#Look at the first few rows of PM10 file
pm10_raw.head()

,Date,Time,PARRAMATTA NORTH SO2 1h average [pphm],PARRAMATTA NORTH PM10 1h average [µg/m³]
0,01/01/2020,01:00,0.0,39.1
1,01/01/2020,02:00,NaN,42.8
2,01/01/2020,03:00,0.1,41.1
3,01/01/2020,04:00,0.1,40.6
4,01/01/2020,05:00,0.0,40.3


In [29]:
#Creating PM10 timestamp to compare with the PM2.5 of Parramatta North
pm10_check = pm10_raw.copy()

# Convert Date using day/month/year format
pm10_check["Date"] = pd.to_datetime(
    pm10_check["Date"],
    format="%d/%m/%Y"
)

pm10_check["Time"] = pm10_check["Time"].astype(str)

# Handle 24:00
midnight_24 = pm10_check["Time"] == "24:00"

pm10_check.loc[midnight_24, "Time"] = "00:00"

pm10_check.loc[midnight_24, "Date"] = (
    pm10_check.loc[midnight_24, "Date"] + pd.Timedelta(days=1)
)

# Create DateTime
pm10_check["DateTime"] = pd.to_datetime(
    pm10_check["Date"].dt.strftime("%d/%m/%Y") + " " +
    pm10_check["Time"],
    format="%d/%m/%Y %H:%M"
)

print("First DateTime:")
print(pm10_check["DateTime"].iloc[0])

print("\nLast DateTime:")
print(pm10_check["DateTime"].iloc[-1])

First DateTime:
2020-01-01 01:00:00

Last DateTime:
2026-07-19 00:00:00


In [30]:
#Check PM10 missingness
pm10_col = "PARRAMATTA NORTH PM10 1h average [µg/m³]"

print("PM10 missing values:")
print(pm10_check[pm10_col].isna().sum())

print("\nPM10 total observations:")
print(len(pm10_check))

print("\nPM10 missing percentage:")
print(
    pm10_check[pm10_col].isna().mean() * 100
)

PM10 missing values:
754

PM10 total observations:
57384

PM10 missing percentage:
1.3139551094381707


In [31]:
#Comparing the missing values gaps between PM2.5 and PM 10 in Parramatta North
start_gap = pd.Timestamp("2023-12-07 05:00:00")
end_gap = pd.Timestamp("2024-01-17 11:00:00")

gap_period = pm10_check[
    (pm10_check["DateTime"] >= start_gap) &
    (pm10_check["DateTime"] <= end_gap)
]

print("Number of PM10 observations in the PM2.5 gap period:")
print(len(gap_period))

print("\nPM10 missing observations during PM2.5 gap period:")
print(gap_period[pm10_col].isna().sum())

print("\nPM10 missing percentage during PM2.5 gap period:")
print(gap_period[pm10_col].isna().mean() * 100)

Number of PM10 observations in the PM2.5 gap period:
991

PM10 missing observations during PM2.5 gap period:
7

PM10 missing percentage during PM2.5 gap period:
0.7063572149344097


##### 1.11 Conversion of Legacy Excel Files
All the .xls files are facing a compatibility issue and hence why we are converting them to .xlsx

In [32]:
#Creating a folder to store the converted .xlsx working files (for the files after conversion)
from pathlib import Path

processed_dir = Path("../data/processed")

processed_dir.mkdir(parents=True, exist_ok=True) # If folder already exists, don't give an error

print("Processed data folder:", processed_dir.resolve())

Processed data folder: C:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone\data\processed


In [33]:
#Test the automated conversion from .xlx to .xlsx into processed folder using a Camden station file 

import win32com.client as win32
from pathlib import Path

# Define raw and processed data directories
raw_dir = Path("../data/raw")
processed_dir = Path("../data/processed")

# Create the processed directory if it does not already exist
processed_dir.mkdir(parents=True, exist_ok=True)

# Test conversion using the Camden PM2.5 file
source_file = raw_dir / "Camden" / "PM10, PM2.5, Ammonia.xls"
output_file = processed_dir / "Camden" / "PM10, PM2.5, Ammonia.xlsx"

# Create the Camden processed folder
output_file.parent.mkdir(parents=True, exist_ok=True)

# Start Excel
excel = win32.DispatchEx("Excel.Application")
excel.Visible = False
excel.DisplayAlerts = False

try:
    # Open the original .xls file
    workbook = excel.Workbooks.Open(str(source_file.resolve()))

    # Save as .xlsx in the processed directory
    workbook.SaveAs(
        str(output_file.resolve()),
        FileFormat=51
    )

    workbook.Close(SaveChanges=False)

    print("Conversion successful!")
    print("Original:", source_file.resolve())
    print("Converted:", output_file.resolve())

finally:
    excel.Quit()

Conversion successful!
Original: C:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone\data\raw\Camden\PM10, PM2.5, Ammonia.xls
Converted: C:\Safa\UTS\YEAR 4\Semester 9 - Spring Session\Engineering Capstone\Sydney-PM25-Capstone\data\processed\Camden\PM10, PM2.5, Ammonia.xlsx


In [34]:
#Finding no. of original .xls files
import win32com.client as win32
from pathlib import Path

# Define raw and processed directories
raw_dir = Path("../data/raw")
processed_dir = Path("../data/processed")

# Find all original .xls files
xls_files = list(raw_dir.rglob("*.xls"))

print("Number of .xls files found:", len(xls_files))

Number of .xls files found: 57


In [35]:
# Convert all original .xls files to .xlsx

converted_files = []
failed_files = []

# Start Excel
excel = win32.DispatchEx("Excel.Application")
excel.Visible = False
excel.DisplayAlerts = False

try:
    for source_file in xls_files:
        
        # Identify the station folder
        station_name = source_file.parent.name
        
        # Create corresponding processed station folder
        station_processed_dir = processed_dir / station_name
        station_processed_dir.mkdir(parents=True, exist_ok=True)
        
        # Create output filename
        output_file = station_processed_dir / f"{source_file.stem}.xlsx"
        
        try:
            # Open original .xls file
            workbook = excel.Workbooks.Open(
                str(source_file.resolve())
            )
            
            # Save as .xlsx
            workbook.SaveAs(
                str(output_file.resolve()),
                FileFormat=51
            )
            
            # Close workbook
            workbook.Close(SaveChanges=False)
            
            converted_files.append({
                "Station": station_name,
                "Original File": source_file.name,
                "Converted File": output_file.name
            })
            
            print(f"Converted: {station_name} / {source_file.name}")
            
        except Exception as e:
            
            failed_files.append({
                "Station": station_name,
                "File": source_file.name,
                "Error": type(e).__name__,
                "Message": str(e)
            })
            
            print(f"FAILED: {station_name} / {source_file.name}")
            print(f"       {type(e).__name__}: {e}")

finally:
    excel.Quit()

print("\nConversion complete.")
print("Successfully converted:", len(converted_files))
print("Failed:", len(failed_files))

Converted: Camden / Air Temperature, Relative Humidity.xls
Converted: Camden / NO2, CO, SO2.xls
Converted: Camden / Ozone, NO.xls
Converted: Camden / PM10, PM2.5, Ammonia.xls
Converted: Camden / Rainfall.xls
Converted: Camden / Wind Speed, Wind Direction.xls
Converted: Lindfield / Air Temperature, Relative Humidity, Rainfall.xls
Converted: Lindfield / NO2, CO, SO2.xls
Converted: Lindfield / Ozone, NO.xls
Converted: Lindfield / PM10, PM2.5, Ammonia.xls
Converted: Lindfield / Wind Speed, Wind Direction.xls
Converted: Liverpool / Air Temperature, Relative Humidity.xls
Converted: Liverpool / NO2, CO.xls
Converted: Liverpool / Ozone, NO.xls
Converted: Liverpool / PM2.5, Ammonia.xls
Converted: Liverpool / Rainfall.xls
Converted: Liverpool / SO2, PM10.xls
Converted: Liverpool / Wind Speed, Wind Direction.xls
Converted: Parramatta North / Air Temperature, Relative Humidity.xls
Converted: Parramatta North / NO2, CO.xls
Converted: Parramatta North / Ozone, NO.xls
Converted: Parramatta North / PM

##### 1.12 PM2.5 Missingness Across Stations

In [36]:
#Creating a table with PM2.5 missing values across all stations
from pathlib import Path
import pandas as pd

# Define the processed data directory
processed_dir = Path("../data/processed")

# Find all processed PM2.5 files
pm25_files = [
    file for file in processed_dir.rglob("*.xlsx")
    if "PM2.5" in file.name
]

print("Number of PM2.5 files found:", len(pm25_files))

# Store results for each station
pm25_results = []

for pm25_file in pm25_files:
    
    station_name = pm25_file.parent.name
    
    try:
        # Read the converted Excel file
        df = pd.read_excel(
            pm25_file,
            header=2
        )
        
        # Find the PM2.5 column
        pm25_columns = [
            col for col in df.columns
            if "PM2.5" in str(col)
        ]
        
        if len(pm25_columns) == 0:
            pm25_results.append({
                "Station": station_name,
                "File": pm25_file.name,
                "Total Observations": len(df),
                "Missing PM2.5": None,
                "Missing Percentage": None,
                "Status": "PM2.5 column not found"
            })
            continue
        
        pm25_col = pm25_columns[0]
        
        total = len(df)
        missing = df[pm25_col].isna().sum()
        missing_percentage = (missing / total) * 100
        
        pm25_results.append({
            "Station": station_name,
            "File": pm25_file.name,
            "Total Observations": total,
            "Missing PM2.5": missing,
            "Missing Percentage": missing_percentage,
            "Status": "OK"
        })
        
    except Exception as e:
        pm25_results.append({
            "Station": station_name,
            "File": pm25_file.name,
            "Total Observations": None,
            "Missing PM2.5": None,
            "Missing Percentage": None,
            "Status": f"Error: {type(e).__name__}"
        })

# Create summary table
pm25_missing_summary = pd.DataFrame(pm25_results)

# Sort by station name
pm25_missing_summary = pm25_missing_summary.sort_values(
    "Station"
).reset_index(drop=True)

pm25_missing_summary

Number of PM2.5 files found: 9


,Station,File,Total Observations,Missing PM2.5,Missing Percentage,Status
0,Camden,"PM10, PM2.5, Ammonia.xlsx",57384,3040.0,5.297644,OK
1,Lindfield,"PM10, PM2.5, Ammonia.xlsx",57384,NaN,NaN,PM2.5 column not found
2,Liverpool,"PM2.5, Ammonia.xlsx",57384,2170.0,3.781542,OK
3,Parramatta North,"PM2.5, Ammonia.xlsx",57384,2393.0,4.170152,OK
4,Penrith,"PM2.5, Ammonia.xlsx",57384,5773.0,10.060296,OK
5,Prospect,"PM2.5, Ammonia.xlsx",57384,3930.0,6.848599,OK
6,Randwick,"PM10, PM2.5, Ammonia.xlsx",57384,1516.0,2.641851,OK
7,Rozelle,"PM2.5, Ammonia.xlsx",57384,1321.0,2.302035,OK
8,Ultimo-UTS,"PM2.5, Ammonia.xlsx",57384,49646.0,86.515405,OK


##### 1.13 Input Variable Files for Retained Stations

In [37]:
#Checking no. of files for the 7 retained stations

#Dropped Lindfield and Ultimo-UTS station because of its missing data
from pathlib import Path
import pandas as pd

# Stations retained for the project
retained_stations = [
    "Camden",
    "Liverpool",
    "Parramatta North",
    "Penrith",
    "Prospect",
    "Randwick",
    "Rozelle"
]

# Directory containing converted Excel files
processed_dir = Path("../data/processed")

# Find all processed Excel files belonging to retained stations
processed_files = []

for station in retained_stations:
    station_dir = processed_dir / station
    
    for file in station_dir.glob("*.xlsx"):
        processed_files.append({
            "Station": station,
            "File": file
        })

print("Number of processed files found:", len(processed_files))

Number of processed files found: 46


##### 1.14 Variable Availability Across Retained Stations

In [38]:
# Identify the variables contained in each processed file

variable_inventory = []

for item in processed_files:
    
    station = item["Station"]
    file = item["File"]
    
    try:
        # Read only the header rows to identify the columns
        df_header = pd.read_excel(
            file,
            header=2,
            nrows=0
        )
        
        # Exclude Date and Time because they are not pollutant/
        # meteorological variables
        variables = [
            column for column in df_header.columns
            if str(column).strip() not in ["Date", "Time"]
        ]
        
        for variable in variables:
            variable_inventory.append({
                "Station": station,
                "File": file.name,
                "Variable": variable
            })
            
    except Exception as e:
        variable_inventory.append({
            "Station": station,
            "File": file.name,
            "Variable": f"ERROR: {type(e).__name__}"
        })

# Create inventory table
variable_inventory_df = pd.DataFrame(variable_inventory)

# Display the inventory
variable_inventory_df

,Station,File,Variable
0,Camden,"Air Temperature, Relative Humidity.xlsx",CAMDEN TEMP 1h average [°C]
1,Camden,"Air Temperature, Relative Humidity.xlsx",CAMDEN HUMID 1h average [%]
2,Camden,"NO2, CO, SO2.xlsx",CAMDEN NO2 1h average [pphm]
3,Camden,"NO2, CO, SO2.xlsx",CAMDEN CO 1h average [ppm]
4,Camden,"Ozone, NO.xlsx",CAMDEN NO 1h average [pphm]
...,...,...,...
76,Rozelle,Rainfall.xlsx,ROZELLE RAIN 1h average [mm/m²]
77,Rozelle,"SO2, PM10.xlsx",ROZELLE SO2 1h average [pphm]
78,Rozelle,"SO2, PM10.xlsx",ROZELLE PM10 1h average [µg/m³]
79,Rozelle,"Wind Speed, Wind Direction.xlsx",ROZELLE WDR 1h average [°]


##### 1.15 Station-Level Variable Availability

In [39]:
#Checking if variables present in each excel file 

# Identify variables directly from the actual Excel column headers

import re

variable_inventory = []

for item in processed_files:
    
    station = item["Station"]
    file = item["File"]
    
    try:
        # Read only the column headers
        df_header = pd.read_excel(
            file,
            header=2,
            nrows=0
        )
        
        for column in df_header.columns:
            
            column_text = str(column).strip()
            
            # Ignore Date and Time columns
            if column_text in ["Date", "Time"]:
                continue
            
            # Extract the variable name from the actual column header.
            # The NSW headers follow the general pattern:
            # STATION VARIABLE 1h average [unit]
            match = re.search(
                r"(.+?)\s+1h average",
                column_text,
                flags=re.IGNORECASE
            )
            
            if match:
                full_variable = match.group(1).strip()
                
                # Remove the station name from the beginning.
                # This handles both single-word and multi-word station names.
                station_prefix = station.upper() + " "
                
                if full_variable.upper().startswith(station_prefix):
                    variable = full_variable[len(station_prefix):].strip()
                else:
                    variable = full_variable
                
                variable_inventory.append({
                    "Station": station,
                    "File": file.name,
                    "Variable": variable,
                    "Original Column": column_text
                })
            
    except Exception as e:
        variable_inventory.append({
            "Station": station,
            "File": file.name,
            "Variable": f"ERROR: {type(e).__name__}",
            "Original Column": ""
        })

# Create inventory DataFrame
variable_inventory_df = pd.DataFrame(variable_inventory)

# Display the complete inventory
variable_inventory_df

,Station,File,Variable,Original Column
0,Camden,"Air Temperature, Relative Humidity.xlsx",TEMP,CAMDEN TEMP 1h average [°C]
1,Camden,"Air Temperature, Relative Humidity.xlsx",HUMID,CAMDEN HUMID 1h average [%]
2,Camden,"NO2, CO, SO2.xlsx",NO2,CAMDEN NO2 1h average [pphm]
3,Camden,"NO2, CO, SO2.xlsx",CO,CAMDEN CO 1h average [ppm]
4,Camden,"Ozone, NO.xlsx",NO,CAMDEN NO 1h average [pphm]
...,...,...,...,...
76,Rozelle,Rainfall.xlsx,RAIN,ROZELLE RAIN 1h average [mm/m²]
77,Rozelle,"SO2, PM10.xlsx",SO2,ROZELLE SO2 1h average [pphm]
78,Rozelle,"SO2, PM10.xlsx",PM10,ROZELLE PM10 1h average [µg/m³]
79,Rozelle,"Wind Speed, Wind Direction.xlsx",WDR,ROZELLE WDR 1h average [°]


##### 1.16 Missingness of Available Variables

In [40]:
# Assess missing values for every available variable
# across the seven retained stations

missingness_results = []

for _, row in variable_inventory_df.iterrows():
    
    station = row["Station"]
    file_name = row["File"]
    variable = row["Variable"]
    original_column = row["Original Column"]
    
    # Locate the corresponding processed file
    file_path = processed_dir / station / file_name
    
    try:
        # Load the dataset
        df = pd.read_excel(
            file_path,
            header=2
        )
        
        # Check that the identified column exists
        if original_column not in df.columns:
            missingness_results.append({
                "Station": station,
                "Variable": variable,
                "Total Observations": len(df),
                "Missing Observations": None,
                "Missing Percentage": None,
                "Status": "Column not found"
            })
            continue
        
        # Calculate missingness
        total = len(df)
        missing = df[original_column].isna().sum()
        missing_percentage = (missing / total) * 100
        
        missingness_results.append({
            "Station": station,
            "Variable": variable,
            "Total Observations": total,
            "Missing Observations": missing,
            "Missing Percentage": missing_percentage,
            "Status": "OK"
        })
        
    except Exception as e:
        missingness_results.append({
            "Station": station,
            "Variable": variable,
            "Total Observations": None,
            "Missing Observations": None,
            "Missing Percentage": None,
            "Status": f"Error: {type(e).__name__}"
        })

# Create the missingness DataFrame
missingness_df = pd.DataFrame(missingness_results)

# Sort for easier interpretation
missingness_df = missingness_df.sort_values(
    ["Station", "Variable"]
).reset_index(drop=True)

# Display the results
missingness_df

,Station,Variable,Total Observations,Missing Observations,Missing Percentage,Status
0,Camden,CO,57384,6062,10.563920,OK
1,Camden,HUMID,57384,920,1.603234,OK
2,Camden,NO,57384,5847,10.189251,OK
3,Camden,NO2,57384,5844,10.184023,OK
4,Camden,OZONE,57384,5334,9.295274,OK
...,...,...,...,...,...,...
76,Rozelle,RAIN,57384,1651,2.877109,OK
77,Rozelle,SO2,57384,4606,8.026628,OK
78,Rozelle,TEMP,57384,862,1.502161,OK
79,Rozelle,WDR,57384,646,1.125749,OK


##### 1.17 Missingness Summary Across Stations and Variables

In [41]:
# Create a compact station × variable missingness table

missingness_matrix = (
    missingness_df
    .pivot(
        index="Station",
        columns="Variable",
        values="Missing Percentage"
    )
)

# Round percentages for easier reading
missingness_matrix = missingness_matrix.round(2)

missingness_matrix

Variable,CO,HUMID,NO,NO2,OZONE,PM10,PM2.5,RAIN,SO2,TEMP,WDR,WSP
Station,,,,,,,,,,,,
Camden,10.56,1.60,10.19,10.18,9.30,2.16,5.30,26.43,NaN,1.58,1.33,1.32
Liverpool,10.88,1.43,6.78,6.78,6.67,1.77,3.78,26.20,6.64,1.36,0.70,0.70
Parramatta North,5.82,0.75,7.41,7.41,4.46,1.31,4.17,35.94,5.60,0.72,0.18,0.18
Penrith,15.72,9.45,13.70,13.70,12.08,9.51,10.06,8.32,13.00,9.40,7.91,7.91
Prospect,7.14,1.36,6.88,6.88,5.55,2.76,6.85,NaN,5.99,1.33,1.25,1.25
Randwick,NaN,2.63,7.62,7.61,5.65,1.64,2.64,42.74,6.96,2.60,3.82,3.82
Rozelle,8.10,1.53,7.21,7.20,6.22,2.16,2.30,2.88,8.03,1.50,1.13,1.13


##### 1.18 Timestamp Continuity Assessment
We are checking if time continuity is there or the entire observation is missing


In [42]:
# Check whether the hourly timestamps are continuous
# using the Parramatta North PM2.5 dataset

pm25_file = (
    processed_dir
    / "Parramatta North"
    / "PM2.5, Ammonia.xlsx"
)

pm25_timestamp_check = pd.read_excel(
    pm25_file,
    header=2
)

# Convert Date to datetime
pm25_timestamp_check["Date"] = pd.to_datetime(
    pm25_timestamp_check["Date"],
    format="%d/%m/%Y"
)

# Convert Time to string
pm25_timestamp_check["Time"] = (
    pm25_timestamp_check["Time"]
    .astype(str)
)

# Handle 24:00
midnight_24 = pm25_timestamp_check["Time"] == "24:00"

pm25_timestamp_check.loc[midnight_24, "Time"] = "00:00"

pm25_timestamp_check.loc[midnight_24, "Date"] = (
    pm25_timestamp_check.loc[midnight_24, "Date"]
    + pd.Timedelta(days=1)
)

# Create timestamp
pm25_timestamp_check["DateTime"] = pd.to_datetime(
    pm25_timestamp_check["Date"].dt.strftime("%d/%m/%Y")
    + " "
    + pm25_timestamp_check["Time"],
    format="%d/%m/%Y %H:%M"
)

# Calculate difference between consecutive timestamps
time_difference = (
    pm25_timestamp_check["DateTime"]
    .diff()
    .dropna()
)

# Count different time intervals
time_difference_counts = (
    time_difference
    .value_counts()
    .sort_index()
)

print("Time intervals between consecutive observations:")
print(time_difference_counts)

print("\nNumber of intervals that are NOT exactly 1 hour:")
print(
    (time_difference != pd.Timedelta(hours=1)).sum()
)

Time intervals between consecutive observations:
DateTime
0 days 01:00:00    57383
Name: count, dtype: int64

Number of intervals that are NOT exactly 1 hour:
0


###### Conclusion from the above code

 ###### The Parramatta North PM2.5 dataset has a continuous hourly timestamp structure. Missing PM2.5 measurements occur as missing values (NaN) within the hourly timeline, rather than as missing rows/timestamps.

##### 1.19 Missing-Value Gap Analysis Across Retained Stations
This step examines the length and distribution of consecutive missing-value periods across the available variables at the seven retained monitoring stations. 

The purpose is to distinguish short isolated gaps from prolonged missing periods before selecting an appropriate missing-data treatment strategy.


In [43]:
# Analyse consecutive missing-value gaps across all retained stations and variables

retained_stations = [
    "Camden",
    "Liverpool",
    "Parramatta North",
    "Penrith",
    "Prospect",
    "Randwick",
    "Rozelle"
]

processed_dir = Path("../data/processed")

gap_results = []

for station in retained_stations:
    station_dir = processed_dir / station

    # Read every processed Excel file for this station
    for file in station_dir.glob("*.xlsx"):

        try:
            df = pd.read_excel(file, header=2)

            # Identify the date and time columns
            date_col = "Date"
            time_col = "Time"

            # Convert Date and Time into one timestamp
            df[date_col] = pd.to_datetime(
                df[date_col],
                format="%d/%m/%Y"
            )

            df[time_col] = df[time_col].astype(str)

            # Handle "24:00" by moving it to the next day
            midnight_24 = df[time_col] == "24:00"

            df.loc[midnight_24, time_col] = "00:00"

            df.loc[midnight_24, date_col] = (
                df.loc[midnight_24, date_col]
                + pd.Timedelta(days=1)
            )

            df["DateTime"] = pd.to_datetime(
                df[date_col].dt.strftime("%d/%m/%Y")
                + " "
                + df[time_col],
                format="%d/%m/%Y %H:%M"
            )

            # Examine every measurement column
            measurement_columns = [
                col for col in df.columns
                if col not in [date_col, time_col, "DateTime"]
            ]

            for variable in measurement_columns:

                missing = df[variable].isna()

                # Identify consecutive groups of missing/non-missing values
                group_id = missing.ne(missing.shift()).cumsum()

                missing_groups = df.loc[missing].groupby(group_id)

                for _, group in missing_groups:

                    start_time = group["DateTime"].iloc[0]
                    end_time = group["DateTime"].iloc[-1]
                    gap_length = len(group)

                    gap_results.append({
                        "Station": station,
                        "Variable": variable,
                        "File": file.name,
                        "Start": start_time,
                        "End": end_time,
                        "Gap Length (hours)": gap_length
                    })

        except Exception as e:
            print(f"Error processing {file}: {e}")

gap_df = pd.DataFrame(gap_results)

print("Total missing-value gaps identified:", len(gap_df))

print("\nFirst few missing-value gaps:")
display(gap_df.head(10))

Total missing-value gaps identified: 80296

First few missing-value gaps:


,Station,Variable,File,Start,End,Gap Length (hours)
0,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-01-07 23:00:00,2020-01-08 11:00:00,13
1,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-02-08 10:00:00,2020-02-08 16:00:00,7
2,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-02-15 18:00:00,2020-02-15 18:00:00,1
3,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-02-16 21:00:00,2020-02-17 06:00:00,10
4,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-02-18 21:00:00,2020-02-19 11:00:00,15
5,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-02-21 10:00:00,2020-02-21 10:00:00,1
6,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-02-26 09:00:00,2020-02-26 09:00:00,1
7,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-03-13 12:00:00,2020-03-13 12:00:00,1
8,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-04-05 22:00:00,2020-04-06 13:00:00,16
9,Camden,CAMDEN TEMP 1h average [°C],"Air Temperature, Relative Humidity.xlsx",2020-06-05 11:00:00,2020-06-05 11:00:00,1


In [44]:
# Summarise missing gaps by station and variable

gap_summary = (
    gap_df
    .groupby(["Station", "Variable"])["Gap Length (hours)"]
    .agg(
        Number_of_Gaps="count",
        Total_Missing_Hours="sum",
        Mean_Gap_Length="mean",
        Median_Gap_Length="median",
        Maximum_Gap_Length="max"
    )
    .reset_index()
)

gap_summary

,Station,Variable,Number_of_Gaps,Total_Missing_Hours,Mean_Gap_Length,Median_Gap_Length,Maximum_Gap_Length
0,Camden,CAMDEN CO 1h average [ppm],2105,6062,2.879810,1.0,2329
1,Camden,CAMDEN HUMID 1h average [%],92,920,10.000000,2.0,163
2,Camden,CAMDEN NO 1h average [pphm],2104,5847,2.778992,1.0,2056
3,Camden,CAMDEN NO2 1h average [pphm],2104,5844,2.777567,1.0,2056
4,Camden,CAMDEN OZONE 1h average [pphm],2139,5334,2.493689,1.0,2056
...,...,...,...,...,...,...,...
76,Rozelle,ROZELLE RAIN 1h average [mm/m²],82,1651,20.134146,2.0,799
77,Rozelle,ROZELLE SO2 1h average [pphm],2164,4606,2.128466,1.0,379
78,Rozelle,ROZELLE TEMP 1h average [°C],90,862,9.577778,2.0,287
79,Rozelle,ROZELLE WDR 1h average [°],71,646,9.098592,2.0,289


###### **Missing-Value Gap Terms**

###### * **Missing-value gap:** A continuous sequence of missing (`NaN`) observations.
###### * **Number of gaps:** Number of separate missing periods.
###### * **Gap length:** Duration of one consecutive missing period, measured in hours.
###### * **Total missing hours:** Total number of missing observations across all gaps.
###### * **Example:** 3 consecutive missing hours = **1 gap**, **3-hour gap**, and **3 total missing hours**.
###### * A dataset can have **many short gaps** or **fewer but very long gaps**, even if their total missing hours are similar.
###### * Gap length is important for deciding how missing values should be handled.


In [45]:
#Making a CSV file of the above summary of missing gaps by station and variable
gap_summary.to_csv(
    "../results/tables/missing_gap_summary.csv",
    index=False
)

print("Saved missing gap summary.")

Saved missing gap summary.


In [46]:
# Categorise missing gaps by duration

def classify_gap(hours):
    if hours == 1:
        return "1 hour"
    elif hours <= 3:
        return "2–3 hours"
    elif hours <= 6:
        return "4–6 hours"
    elif hours <= 24:
        return "7–24 hours"
    elif hours <= 168:
        return "25–168 hours"
    else:
        return ">168 hours"


gap_df["Gap Category"] = gap_df["Gap Length (hours)"].apply(classify_gap)

gap_category_summary = (
    gap_df["Gap Category"]
    .value_counts()
    .reindex([
        "1 hour",
        "2–3 hours",
        "4–6 hours",
        "7–24 hours",
        "25–168 hours",
        ">168 hours"
    ])
    .fillna(0)
    .astype(int)
)

print("Number of missing gaps by duration:")
print(gap_category_summary)

Number of missing gaps by duration:
Gap Category
1 hour          72031
2–3 hours        4445
4–6 hours        1162
7–24 hours       1619
25–168 hours      962
>168 hours         77
Name: count, dtype: int64


##### 1.20 Investigation of Long Missing-Value Gaps
This step identifies prolonged missing-value periods (greater than 168 hours) to determine whether long gaps are isolated to individual variables or occur across multiple variables at the same station and time period. This information will be used to inform the missing-data treatment during preprocessing.

In [47]:
# Identify all prolonged missing-value gaps (> 168 hours)

long_gaps = (
    gap_df[gap_df["Gap Length (hours)"] > 168]
    .sort_values(
        ["Station", "Start", "Variable"]
    )
    .reset_index(drop=True)
)

print("Number of gaps longer than 168 hours:", len(long_gaps))

display(long_gaps)

Number of gaps longer than 168 hours: 77


,Station,Variable,File,Start,End,Gap Length (hours),Gap Category
0,Camden,CAMDEN RAIN 1h average [mm/m²],Rainfall.xlsx,2020-01-01 01:00:00,2021-08-26 16:00:00,14488,>168 hours
1,Camden,CAMDEN CO 1h average [ppm],"NO2, CO, SO2.xlsx",2023-06-02 15:00:00,2023-09-07 15:00:00,2329,>168 hours
2,Camden,CAMDEN NO 1h average [pphm],"Ozone, NO.xlsx",2023-06-02 15:00:00,2023-06-10 02:00:00,180,>168 hours
3,Camden,CAMDEN NO2 1h average [pphm],"NO2, CO, SO2.xlsx",2023-06-02 15:00:00,2023-06-10 02:00:00,180,>168 hours
4,Camden,CAMDEN OZONE 1h average [pphm],"Ozone, NO.xlsx",2023-06-02 15:00:00,2023-06-10 02:00:00,180,>168 hours
...,...,...,...,...,...,...,...
72,Rozelle,ROZELLE WDR 1h average [°],"Wind Speed, Wind Direction.xlsx",2023-01-04 14:00:00,2023-01-16 14:00:00,289,>168 hours
73,Rozelle,ROZELLE WSP 1h average [m/s],"Wind Speed, Wind Direction.xlsx",2023-01-04 14:00:00,2023-01-16 14:00:00,289,>168 hours
74,Rozelle,ROZELLE SO2 1h average [pphm],"SO2, PM10.xlsx",2025-05-22 08:00:00,2025-06-07 02:00:00,379,>168 hours
75,Rozelle,ROZELLE CO 1h average [ppm],"NO2, CO.xlsx",2025-08-27 02:00:00,2025-09-08 02:00:00,289,>168 hours


In [48]:
## Save the identified long missing-value gaps to a CSV file
# so they can be reviewed later and used to support preprocessing decisions.
long_gaps.to_csv(
    "../results/tables/long_missing_gaps.csv",
    index=False
)

print("Saved long missing gaps into a csv file under 'results' in 'tables' .")

Saved long missing gaps into a csv file under 'results' in 'tables' .


##### 1.21 Impact of Prolonged Missing-Value Gaps
This step compares the total number of missing observations with the number contained in prolonged gaps greater than 168 hours. 

The comparison is used to distinguish isolated or short missing observations from systematic periods of missing data before preprocessing.

In [49]:
# Calculate how much of the missing data is contained in prolonged gaps

long_gap_hours = (
    long_gaps
    .groupby(["Station", "Variable"])["Gap Length (hours)"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "Gap Length (hours)": "Long_Gap_Missing_Hours"
        }
    )
)

missing_impact_summary = gap_summary.merge(
    long_gap_hours,
    on=["Station", "Variable"],
    how="left"
)

missing_impact_summary["Long_Gap_Missing_Hours"] = (
    missing_impact_summary["Long_Gap_Missing_Hours"]
    .fillna(0)
)

missing_impact_summary["Percentage_of_Missing_in_Long_Gaps"] = (
    missing_impact_summary["Long_Gap_Missing_Hours"]
    / missing_impact_summary["Total_Missing_Hours"]
    * 100
)

missing_impact_summary = (
    missing_impact_summary
    .sort_values(
        "Percentage_of_Missing_in_Long_Gaps",
        ascending=False
    )
    .reset_index(drop=True)
)

display(missing_impact_summary)

,Station,Variable,Number_of_Gaps,Total_Missing_Hours,Mean_Gap_Length,Median_Gap_Length,Maximum_Gap_Length,Long_Gap_Missing_Hours,Percentage_of_Missing_in_Long_Gaps
0,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],34,20625,606.617647,1.0,20487,20487.0,99.330909
1,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],73,15035,205.958904,1.0,14800,14800.0,98.436980
2,Randwick,RANDWICK RAIN 1h average [mm/m²],36,24524,681.222222,3.5,24037,24037.0,98.014190
3,Penrith,PENRITH WSP 1h average [m/s],46,4538,98.652174,1.0,4427,4427.0,97.553989
4,Penrith,PENRITH WDR 1h average [°],46,4538,98.652174,1.0,4427,4427.0,97.553989
...,...,...,...,...,...,...,...,...,...
76,Randwick,RANDWICK NO2 1h average [pphm],2196,4366,1.988160,1.0,145,0.0,0.000000
77,Randwick,RANDWICK NO 1h average [pphm],2196,4371,1.990437,1.0,145,0.0,0.000000
78,Prospect,PROSPECT NO2 1h average [pphm],2229,3946,1.770301,1.0,168,0.0,0.000000
79,Prospect,PROSPECT SO2 1h average [pphm],2210,3440,1.556561,1.0,168,0.0,0.000000


In [50]:
# Save the above missing-data impact summary

missing_impact_summary.to_csv(
    "../results/tables/missing_impact_summary.csv",
    index=False
)

print("Saved missing-data impact summary under 'results' in 'tables'.")

Saved missing-data impact summary under 'results' in 'tables'.


##### 1.22 Physical Plausibility and Data-Quality Checks

This step checks the available variables for physically impossible or clearly invalid values before preprocessing. Extreme but physically plausible observations are retained because they may represent genuine pollution events rather than measurement errors.

In [51]:
# Check minimum and maximum values for each available variable
# across the seven retained stations.

import pandas as pd
from pathlib import Path

retained_stations = [
    "Camden",
    "Liverpool",
    "Parramatta North",
    "Penrith",
    "Prospect",
    "Randwick",
    "Rozelle"
]

processed_dir = Path("../data/processed")

plausibility_summary = []

for station in retained_stations:
    station_dir = processed_dir / station

    for file in station_dir.glob("*.xlsx"):

        try:
            df = pd.read_excel(file, header=2)

            measurement_columns = [
                col for col in df.columns
                if col not in ["Date", "Time"]
            ]

            for variable in measurement_columns:

                numeric_values = pd.to_numeric(
                    df[variable],
                    errors="coerce"
                )

                plausibility_summary.append({
                    "Station": station,
                    "Variable": variable,
                    "Minimum": numeric_values.min(),
                    "Maximum": numeric_values.max(),
                    "Missing": numeric_values.isna().sum(),
                    "Valid_Values": numeric_values.notna().sum()
                })

        except Exception as e:
            print(f"Error processing {file}: {e}")

plausibility_df = pd.DataFrame(plausibility_summary)

display(plausibility_df)

,Station,Variable,Minimum,Maximum,Missing,Valid_Values
0,Camden,CAMDEN TEMP 1h average [°C],-1.6,45.2,904,56480
1,Camden,CAMDEN HUMID 1h average [%],9.3,103.9,920,56464
2,Camden,CAMDEN NO2 1h average [pphm],-0.1,4.1,5844,51540
3,Camden,CAMDEN CO 1h average [ppm],-0.2,4.9,6062,51322
4,Camden,CAMDEN NO 1h average [pphm],-0.2,6.0,5847,51537
...,...,...,...,...,...,...
76,Rozelle,ROZELLE RAIN 1h average [mm/m²],0.0,49.8,1651,55733
77,Rozelle,ROZELLE SO2 1h average [pphm],-0.2,3.9,4606,52778
78,Rozelle,ROZELLE PM10 1h average [µg/m³],-10.0,420.2,1240,56144
79,Rozelle,ROZELLE WDR 1h average [°],0.0,360.0,646,56738


In [52]:
#Saving the above table to results

plausibility_df.to_csv(
    "../results/tables/variable_plausibility_summary.csv",
    index=False
)

print("Saved variable plausibility summary.")

Saved variable plausibility summary.


##### 1.22.1 Checking for invalid observations

###### The previous table showed the minimum and maximum values for each variable. This step checks the data more directly for observations that are outside reasonable physical or measurement ranges.

###### These observations will not be removed at this stage. They will be converted to missing values during preprocessing if they are confirmed to be invalid.

###### Very large but potentially genuine pollution measurements are kept for further investigation rather than being removed only because they are extreme.

In [53]:
# These checks are used for values that are clearly outside
# a reasonable physical or measurement range.

physical_rules = {
    "HUMID": lambda x: (x < 0) | (x > 100),
    "PM2.5": lambda x: x < 0,
    "PM10": lambda x: x < 0,
    "RAIN": lambda x: x < 0,
    "WDR": lambda x: (x < 0) | (x > 360),
    "WSP": lambda x: x < 0
}

print("Physical plausibility rules have been defined.")

Physical plausibility rules have been defined.


In [54]:
# Check the retained stations for observations that clearly
# fall outside the physical or measurement ranges.

invalid_summary = []

for station in retained_stations:
    station_dir = processed_dir / station

    for file in station_dir.glob("*.xlsx"):

        try:
            df = pd.read_excel(file, header=2)

            measurement_columns = [
                col for col in df.columns
                if col not in ["Date", "Time"]
            ]

            for variable in measurement_columns:

                values = pd.to_numeric(
                    df[variable],
                    errors="coerce"
                )

                rule = None

                # Find the rule that matches this variable.
                for variable_name in physical_rules:
                    if variable_name in variable:
                        rule = physical_rules[variable_name]
                        break

                if rule is None:
                    continue

                invalid = rule(values)

                invalid_summary.append({
                    "Station": station,
                    "Variable": variable,
                    "File": file.name,
                    "Invalid_Observations": invalid.sum()
                })

        except Exception as e:
            print(f"Error processing {file}: {e}")

invalid_df = pd.DataFrame(invalid_summary)

display(invalid_df)

,Station,Variable,File,Invalid_Observations
0,Camden,CAMDEN HUMID 1h average [%],"Air Temperature, Relative Humidity.xlsx",2291
1,Camden,CAMDEN PM10 1h average [µg/m³],"PM10, PM2.5, Ammonia.xlsx",944
2,Camden,CAMDEN PM2.5 1h average [µg/m³],"PM10, PM2.5, Ammonia.xlsx",6901
3,Camden,CAMDEN RAIN 1h average [mm/m²],Rainfall.xlsx,0
4,Camden,CAMDEN WDR 1h average [°],"Wind Speed, Wind Direction.xlsx",0
5,Camden,CAMDEN WSP 1h average [m/s],"Wind Speed, Wind Direction.xlsx",0
6,Liverpool,LIVERPOOL HUMID 1h average [%],"Air Temperature, Relative Humidity.xlsx",475
7,Liverpool,LIVERPOOL PM2.5 1h average [µg/m³],"PM2.5, Ammonia.xlsx",4447
8,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],Rainfall.xlsx,0
9,Liverpool,LIVERPOOL PM10 1h average [µg/m³],"SO2, PM10.xlsx",437


In [55]:
# Save the clearly invalid observations so we can refer to them
# when we start cleaning the data in the next notebook.

invalid_df.to_csv(
    "../results/tables/clearly_invalid_observations.csv",
    index=False
)

print("Saved clearly invalid observations.")

Saved clearly invalid observations.


##### 1.22.2 Checking for suspicious negative pollutant readings

Some pollutant variables contain negative readings. Since pollutant concentrations cannot physically be negative, these observations are suspicious.

They are kept separate from the clearly invalid observations because the inspection stage alone does not establish why these negative readings occur.

These values will be reviewed during preprocessing before deciding how they should be handled.

In [56]:
# Check the pollutant variables for negative readings.
# These are flagged for further investigation rather than removed now.

# calculates the number of negative observations for every pollutant variable and creates: negative_df
# It contains all pollutant variables, including ones with 0 negative observations

negative_summary = []

for station in retained_stations:
    station_dir = processed_dir / station

    for file in station_dir.glob("*.xlsx"):

        try:
            df = pd.read_excel(file, header=2)

            measurement_columns = [
                col for col in df.columns
                if col not in ["Date", "Time"]
            ]

            for variable in measurement_columns:

                pollutant_name = None

                # Check the variable names in this order so that
                # PM2.5 and NO2 are matched correctly.
                for variable_name in [
                    "PM2.5",
                    "PM10",
                    "NO2",
                    "OZONE",
                    "SO2",
                    "CO",
                    "NO"
                ]:
                    if variable_name in variable:
                        pollutant_name = variable_name
                        break

                if pollutant_name is None:
                    continue

                values = pd.to_numeric(
                    df[variable],
                    errors="coerce"
                )

                negative_count = (values < 0).sum()

                negative_summary.append({
                    "Station": station,
                    "Variable": variable,
                    "File": file.name,
                    "Negative_Observations": negative_count
                })

        except Exception as e:
            print(f"Error processing {file}: {e}")

negative_df = pd.DataFrame(negative_summary)

display(negative_df)

,Station,Variable,File,Negative_Observations
0,Camden,CAMDEN NO2 1h average [pphm],"NO2, CO, SO2.xlsx",21
1,Camden,CAMDEN CO 1h average [ppm],"NO2, CO, SO2.xlsx",7299
2,Camden,CAMDEN NO 1h average [pphm],"Ozone, NO.xlsx",4444
3,Camden,CAMDEN OZONE 1h average [pphm],"Ozone, NO.xlsx",29
4,Camden,CAMDEN PM10 1h average [µg/m³],"PM10, PM2.5, Ammonia.xlsx",944
5,Camden,CAMDEN PM2.5 1h average [µg/m³],"PM10, PM2.5, Ammonia.xlsx",6901
6,Liverpool,LIVERPOOL NO2 1h average [pphm],"NO2, CO.xlsx",2119
7,Liverpool,LIVERPOOL CO 1h average [ppm],"NO2, CO.xlsx",28387
8,Liverpool,LIVERPOOL NO 1h average [pphm],"Ozone, NO.xlsx",2069
9,Liverpool,LIVERPOOL OZONE 1h average [pphm],"Ozone, NO.xlsx",33


In [57]:
# Only show variables where negative readings were found.
# These will be investigated further during preprocessing.

#This step takes negative_df and filters it to show only variables where negative observations actually exist
# filter + sort from the above step basically 

negative_flagged = negative_df[
    negative_df["Negative_Observations"] > 0
].copy()

negative_flagged = (
    negative_flagged
    .sort_values("Negative_Observations", ascending=False)
    .reset_index(drop=True)
)

display(negative_flagged)

,Station,Variable,File,Negative_Observations
0,Liverpool,LIVERPOOL CO 1h average [ppm],"NO2, CO.xlsx",28387
1,Penrith,PENRITH CO 1h average [ppm],"NO2, CO.xlsx",16458
2,Prospect,PROSPECT CO 1h average [ppm],"NO2, CO.xlsx",13187
3,Parramatta North,PARRAMATTA NORTH NO 1h average [pphm],"Ozone, NO.xlsx",8301
4,Camden,CAMDEN CO 1h average [ppm],"NO2, CO, SO2.xlsx",7299
5,Camden,CAMDEN PM2.5 1h average [µg/m³],"PM10, PM2.5, Ammonia.xlsx",6901
6,Prospect,PROSPECT NO 1h average [pphm],"Ozone, NO.xlsx",5167
7,Prospect,PROSPECT PM2.5 1h average [µg/m³],"PM2.5, Ammonia.xlsx",5056
8,Penrith,PENRITH PM2.5 1h average [µg/m³],"PM2.5, Ammonia.xlsx",4660
9,Randwick,RANDWICK PM2.5 1h average [µg/m³],"PM10, PM2.5, Ammonia.xlsx",4653


In [58]:
# Save the suspicious negative pollutant readings so they
# can be referred to during preprocessing.

negative_flagged.to_csv(
    "../results/tables/suspicious_negative_pollutant_readings.csv",
    index=False
)

print("Saved suspicious negative pollutant readings.")

Saved suspicious negative pollutant readings.


###### **Interpretation**

###### The negative-value check identified negative readings in several pollutant variables across the retained stations.

###### Negative PM2.5 and PM10 readings are clearly outside their physical concentration range and were already identified in the physical plausibility checks above. Negative readings were also observed for several gaseous pollutants, including NO2, CO, NO, OZONE and SO2. These observations are recorded for further investigation during preprocessing rather than being automatically removed at this stage.

###### Very large positive pollutant values have not been removed based only on their magnitude, as they may represent genuine pollution events.

###### No observations are removed during the data inspection stage. The identified issues will be addressed when the preprocessing decisions are made.